In [1]:
import sys
from pathlib import Path
import importlib
import logging

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
SRC_DIR = str(Path(PROJECT_DIR) / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import multiomic_transformer.utils.data_formatter as data_formatter
import multiomic_transformer.utils.experiment_handler as experiment_handler
from multiomic_transformer.models import model_simplified

logging.basicConfig(level=logging.INFO, format="%(message)s")

DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
GROUND_TRUTH_DIR = PROJECT_DIR / "data" / "ground_truth_files"
PROCESSED_DATA_DIR = DATA_DIR / "PROCESSED_DATA"
TRAINING_CACHE_DIR = DATA_DIR / "TRAINING_CACHE"

In [18]:
importlib.reload(data_formatter)

# Path to the project directory (same as Git repository root)
project_dir = Path(PROJECT_DIR)
DATA_DIR=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/")
SAMPLE_NAME="sample_1"

# Path to the training output directory. Used to store the preprocessing config
output_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS")

# Name of the dataset / experiment to run
experiment_name = "K562_sample_1_test_peaks_conv_only"
PROCESSED_DATA_DIR=Path(f"{DATA_DIR}/PROCESSED_DATA/{experiment_name}")
TRAINING_CACHE_DIR=Path(f"{DATA_DIR}/TRAINING_DATA_CACHE/{experiment_name}")

# Organism code for the dataset. Supports either "mm10" or "hg38"
organism_code = "hg38"

# List of samples in the training datset. 
# Each of these should have its own subdirectory in the processed data directory
sample_names = ["sample_1"]

# List of chromosomes. Used to split the data by chromsome for caching and training.
# Should be in the format "chr1", "chr2", etc. and should match the chromosome names in the processed data files.
chrom_list = [f"chr{i}" for i in range(1, 23)] + ["chrX"]

tdf = data_formatter.TrainingDataFormatter(
    project_dir=project_dir,
    experiment_name=experiment_name,
    processed_data_dir=PROCESSED_DATA_DIR,
    training_data_cache=TRAINING_CACHE_DIR,
    organism_code=organism_code,
    sample_names=sample_names,
    chrom_list=chrom_list,
    output_dir=output_dir / experiment_name,
)
tdf.chrom_list = chrom_list
tdf.window_size = 100

# Verify that the data cache files exist. If not, this method will create them.
tdf.create_or_load_data_cache(sample_name=tdf.sample_names[0], force_recalculate=False)

Loaded existing settings from /gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA/K562_sample_1_test_peaks_conv_only/settings.json
[0/4] Creating or loading pseudobulk data

[1/4] Creating or loading peak BED file

[2/4] Creating or loading peak-to-target gene distance file
  - Loading saved peak to TSS distance file

[3/4] Aggregating sample-level pseudobulk datasets into global dataset
  - Found existing global and per-chrom pseudobulk; loading...

[4/4] Creating or loading chromosome-specific data files for model training
Initialized TF vocab with 436 entries
Initialized TG vocab with 6126 entries
  - Loading genome-wide genomic windows
  - Number of chromosomes: 23: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX']
  - Processing chromosomes for dataset: K562_sample_1_test_peaks_conv_only
Creating chromos

In [19]:
num_missing_files, missing_file_dict = tdf.check_cached_file_exist()
logging.info(f"Missing {num_missing_files} cached files")

All required files are present.
Missing 0 cached files


In [20]:
importlib.reload(experiment_handler)

logging.info("Initializing ExperimentHandler...")
exp = experiment_handler.ExperimentHandler(
    training_data_formatter=tdf,
    experiment_dir="/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/",
    model_num=1,
    silence_warnings=False,
)

logging.info("Creating dataset...")
exp.create_multichrom_dataset(max_cached=50)

logging.info("Preparing DataLoader...")
train_loader, val_loader, test_loader = exp.prepare_dataloader(
    batch_size=32,
    num_workers=8
)

logging.info("Creating scalers...")
exp.create_scalers(train_loader, max_batches=100)

atac_wins, tf_tensor, tg_tensor, bias, tf_ids, tg_ids = train_loader.dataset[0]
print("\nATAC windows shape:", atac_wins.shape)
print("TF tensor shape:", tf_tensor.shape)
print("TG tensor shape:", tg_tensor.shape)
print("Bias shape:", bias.shape)
print("TF IDs shape:", tf_ids.shape)
print("TG IDs shape:", tg_ids.shape)

Initializing ExperimentHandler...
Creating dataset...
Preparing DataLoader...
Creating scalers...



ATAC windows shape: torch.Size([735, 1])
TF tensor shape: torch.Size([436])
TG tensor shape: torch.Size([586])
Bias shape: torch.Size([586, 735])
TF IDs shape: torch.Size([436])
TG IDs shape: torch.Size([586])


In [21]:
importlib.reload(model_simplified)
window_downsampler = model_simplified.WindowDownsampler(
    in_channels=1, 
    layer_1_kernel_size=8, 
    layer_1_stride=4,
    intermediate_channels=2*exp.d_model,
    layer_2_kernel_size=4,
    layer_2_stride=2,
    out_channels=exp.d_model,
    )

window_downsampler.to(exp.device)
atac_wins = atac_wins.to(exp.device, non_blocking=True)

wins_downsampled = window_downsampler(atac_wins)
print("\nDownsampled ATAC windows shape:", wins_downsampled.shape)


Downsampled ATAC windows shape: torch.Size([1, 90, 128])


In [22]:
importlib.reload(model_simplified)

tf_vocab_size = int(exp.dataset.tf_ids.numel())
tg_vocab_size = int(exp.dataset.tg_ids.numel())

exp.model = model_simplified.MultiomicTransformer(
    d_model=exp.d_model,
    num_heads=exp.num_heads,
    num_layers=exp.num_layers,
    d_ff=exp.d_ff,
    dropout=exp.dropout,
    tf_vocab_size=tf_vocab_size,
    tg_vocab_size=tg_vocab_size,
    use_bias=False,
    bias_scale=exp.bias_scale,
    layer_1_kernel_size=8,
    layer_1_stride=4,
    layer_2_kernel_size=4,
    layer_2_stride=2
).to(exp.device)   

In [12]:
exp.load_model()

# # Runs model training and returns the trained model.
# logging.info("Training model")
# model = exp.train(
#     train_loader=train_loader, 
#     val_loader=val_loader, 
#     num_epochs=50,
#     max_batches=None,
#     save_every_n_epochs=10,
#     monitor_gpu_memory=True,
#     profile_batches=True,
#     allow_overwrite=True,
#     silence_tqdm=True,
#     )

MultiomicTransformer(
  (tf_identity_emb): Embedding(1465, 128)
  (tg_query_emb): Embedding(26611, 128)
  (window_downsampler): WindowDownsampler(
    (net): Sequential(
      (0): Conv1d(1, 256, kernel_size=(8,), stride=(4,))
      (1): GELU(approximate='none')
      (2): Conv1d(256, 128, kernel_size=(4,), stride=(2,))
    )
  )
  (tf_expr_dense_input_layer): Sequential(
    (0): Linear(in_features=1, out_features=512, bias=True)
    (1): SiLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=False)
    (4): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (posenc): PositionalEmbedding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dro

In [23]:
# # Runs gradient attribution to calculate the gradients between each TF input and each TG output.
logging.info("\nRunning Gradient Attribution")
exp.load_model()
exp.run_gradient_attribution(
    test_loader,
    max_batches=None,
    max_tgs_per_batch=None,
    )

exp.grn = exp.load_grn()

# Loads a ground truth file with columns "Source" and "Target" for TF-TG interactions.
logging.info("Loading ground truth datasets...")
GROUND_TRUTH_DIR = Path(PROJECT_DIR) / "data" / "ground_truth_files"
gt_by_dataset_dict = {
    "ChIP-Atlas iPSC (1 Mb)": exp.load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_iPSC_1mb.csv"),
}

# Calculates the AUROC of the predicted GRN against multiple ground truth datasets.
logging.info("\nCalculating AUROC")
auroc_df = exp.calculate_auroc_all_sample_gts(exp.grn, gt_by_dataset_dict)     
logging.info(f"Pooled Median AUROC: {auroc_df['pooled_median_auroc'].iloc[0]:.3f}")       
logging.info(f"Per-TF Median AUROC: {auroc_df['per_tf_median_auroc'].iloc[0]:.3f}")

exp.save_handler()


Running Gradient Attribution
Gradient attributions: 100%|███████████████████████████████████| 69/69 [00:37<00:00,  1.82batches/s]
Loading ground truth datasets...

Calculating AUROC
Pooled Median AUROC: 0.493
Per-TF Median AUROC: 0.489
